# 01 · Train one configuration

Trains one generator with the Pix2Pix setup (PatchGAN discriminator, road-weighted L1
plus a weak adversarial term), keeps the checkpoint with the best validation buffered IoU,
and writes it to `<OUT_ROOT>/<run name>/generator_best.weights.h5`.

One configuration per run: edit `CFG` in section 2, then Run All. Final numbers come from
`02_evaluate.ipynb`, which rescores every checkpoint under one protocol; the test IoU printed
at the end of this notebook is a quick check at a fixed threshold without TTA.

**Configurations used in the thesis** (everything else at the defaults below):

| Experiment | `ARCH` | `ADV_WEIGHT` | `ROAD_WEIGHT` | `IMG_SIZE` | `SEED` |
|---|---|---|---|---|---|
| Generator comparison | each of the five | 0.3 | 10 | 256 | 1907096 and 2024 |
| Ablation: no adversarial term | `unet` | 0.0 | 10 | 256 | 1907096 |
| Ablation: full adversarial term | `unet` | 1.0 | 10 | 256 | 1907096 |
| Ablation: no road weighting | `unet` | 0.3 | 0 | 256 | 1907096 |
| Ablation: 512 input | `unet` | 0.3 | 10 | 512 (`BATCH` 2) | 1907096 |

At 512 the output is resized to 256 before scoring, so every run is evaluated on the same grid.
The segmentation-head runs are trained with `01b_train_segmentation.ipynb`.

## 1. Imports

In [ ]:
import csv, json, math, time, random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

from sat2map import models, data, training, evaluation as ev, runs

print("TensorFlow", tf.__version__)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Configuration

The only cell to edit between runs. `SUBSET` caps the number of training files for
data-scaling runs (0 uses everything); when it is set, the epoch count is scaled up so every
subset sees about the same number of optimisation steps.

In [ ]:
CFG = dict(
    # architecture
    ARCH        = "unet_mid",   # unet | unet_bottleneck | unet_mid | resnet9 | attn_unet
    POS_EMBED   = True,         # learned positional embedding in the transformer block
    NUM_HEADS   = 4,
    FF_DIM      = 1024,

    # data
    IMG_SIZE    = 256,          # 256 or 512
    SUBSET      = 0,            # 0 = all training files

    # loss
    ADV_WEIGHT  = 0.3,          # 0.0 = L1 only, 0.3 = weak (default), 1.0 = full
    L1_WEIGHT   = 100.0,
    ROAD_WEIGHT = 10.0,         # 0.0 gives plain unweighted MAE
    LOSS_THRESH = 0.98,         # target road mask inside the loss

    # training
    BASE_EPOCHS = 30,           # epochs at the full training set
    BATCH       = 4,            # use 2 at IMG_SIZE 512
    LR          = 2e-4,
    BETA_1      = 0.5,
    SEED        = 1907096,      # thesis runs: 1907096 and 2024

    # checkpoint selection
    EVAL_SIZE   = 256,          # everything is scored at this resolution
    BUFFER_PX   = 2,            # dilation tolerance at EVAL_SIZE
    GEN_THRESH  = 0.90,         # generated pixel counted as road
    TAR_THRESH  = 0.90,         # target pixel counted as road
    EVAL_EVERY  = 2,            # epochs between validation checks
    TRAIN_EVAL_N = 100,         # training tiles used to measure the train/val gap
)

DATA_ROOT = "../data/Dataset"   # folder holding train/ val/ test/
OUT_ROOT  = "../runs"           # each run writes to OUT_ROOT/<run name>/

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); tf.random.set_seed(CFG["SEED"])
SIZE = CFG["IMG_SIZE"]

## 3. Data

In [ ]:
root = Path(DATA_ROOT)
if not (root / "train").is_dir():
    parent = root.parent if root.parent.exists() else Path(".")
    print("No train/ folder under", root, "\nFolders under", parent, ":")
    for p in sorted(parent.rglob("*")):
        if p.is_dir():
            print("  ", p)
    raise SystemExit("Fix DATA_ROOT in section 2")

ALL_TRAIN  = data.file_list(root, "train")
VAL_FILES  = data.file_list(root, "val")
TEST_FILES = data.file_list(root, "test")

if CFG["SUBSET"]:
    TRAIN_FILES = ALL_TRAIN[:CFG["SUBSET"]]
    CFG["EPOCHS"] = int(round(CFG["BASE_EPOCHS"] * len(ALL_TRAIN) / len(TRAIN_FILES)))
else:
    TRAIN_FILES = ALL_TRAIN
    CFG["EPOCHS"] = CFG["BASE_EPOCHS"]
CFG["N_TRAIN"] = len(TRAIN_FILES)
CFG["RUN_NAME"] = runs.run_name(CFG["ARCH"], SIZE, CFG["ADV_WEIGHT"],
                                CFG["ROAD_WEIGHT"], CFG["N_TRAIN"], CFG["SEED"])

OUT = Path(OUT_ROOT) / CFG["RUN_NAME"]
OUT.mkdir(parents=True, exist_ok=True)

train_ds      = data.make_ds(TRAIN_FILES, SIZE, CFG["BATCH"], training=True, seed=CFG["SEED"])
val_ds        = data.make_ds(VAL_FILES, SIZE)
test_ds       = data.make_ds(TEST_FILES, SIZE)
train_eval_ds = data.make_ds(TRAIN_FILES[:CFG["TRAIN_EVAL_N"]], SIZE)

print("train %d of %d   val %d   test %d" % (len(TRAIN_FILES), len(ALL_TRAIN),
                                             len(VAL_FILES), len(TEST_FILES)))
print("%d epochs, %d steps per epoch" % (CFG["EPOCHS"],
                                        math.ceil(len(TRAIN_FILES) / CFG["BATCH"])))
print("run:", CFG["RUN_NAME"])

Sanity check: roads are drawn pure white by the collection styling, so the road fraction
of the targets should sit around 3% (median of the collection manifest's `map_detail`).
A value near zero means the palette assumption is broken and nothing downstream is meaningful.

In [ ]:
fr = np.array([float(tf.reduce_mean(tf.cast(data.road_mask_tf(t, CFG["LOSS_THRESH"]), tf.float32)))
               for _, t in train_ds.unbatch().take(200)])
print("road pixel fraction over 200 training targets: mean %.4f  median %.4f  min %.4f  max %.4f"
      % (fr.mean(), np.median(fr), fr.min(), fr.max()))

## 4. Models

In [ ]:
generator = models.make_generator(CFG["ARCH"], SIZE, "rgb", CFG["NUM_HEADS"],
                                  CFG["FF_DIM"], CFG["POS_EMBED"])
discriminator = models.build_discriminator(SIZE)

print("%s generator: %.2fM params" % (CFG["ARCH"], generator.count_params() / 1e6))
for layer in generator.layers:
    if isinstance(layer, models.TransformerBlock):
        print("  transformer at %s: %dx%d = %d tokens"
              % (layer.name, layer.h, layer.w, layer.h * layer.w))
print("discriminator: %.2fM params, patch output %s"
      % (discriminator.count_params() / 1e6, discriminator.output_shape))

g_opt = tf.keras.optimizers.Adam(CFG["LR"], beta_1=CFG["BETA_1"])
d_opt = tf.keras.optimizers.Adam(CFG["LR"], beta_1=CFG["BETA_1"])
train_step = training.make_train_step(generator, discriminator, g_opt, d_opt,
                                      CFG["ADV_WEIGHT"], CFG["L1_WEIGHT"],
                                      CFG["ROAD_WEIGHT"], CFG["LOSS_THRESH"])

def quick_eval(ds):
    return training.quick_eval(generator, ds, CFG["EVAL_SIZE"], CFG["GEN_THRESH"],
                               CFG["TAR_THRESH"], CFG["BUFFER_PX"])

## 5. Train

Logs to `history.csv` every epoch and keeps the generator weights with the best validation
buffered IoU. Training IoU on a fixed subset is logged alongside so the gap is visible during the run.

In [ ]:
log_path, best_path = OUT / "history.csv", OUT / "generator_best.weights.h5"
best_iou, best_epoch, history = -1.0, 0, []
t0 = time.time()

for epoch in range(1, CFG["EPOCHS"] + 1):
    tot, n = np.zeros(4), 0
    for src, tar in train_ds:
        tot += [float(v) for v in train_step(src, tar)]
        n += 1
    tot /= n

    row = dict(epoch=epoch, g_total=tot[0], g_adv=tot[1], g_l1=tot[2], d_loss=tot[3],
               train_strict=np.nan, train_buf=np.nan, val_strict=np.nan, val_buf=np.nan,
               minutes=(time.time() - t0) / 60.0)

    if epoch % CFG["EVAL_EVERY"] == 0 or epoch == CFG["EPOCHS"]:
        tr_s, tr_b = quick_eval(train_eval_ds)
        va_s, va_b = quick_eval(val_ds)
        row.update(train_strict=tr_s, train_buf=tr_b, val_strict=va_s, val_buf=va_b)
        marker = ""
        if va_b > best_iou:
            best_iou, best_epoch = va_b, epoch
            generator.save_weights(str(best_path))
            marker = "  <- best"
        print("epoch %3d  g %.4f  d %.4f | train %.4f  val %.4f  gap %.4f%s"
              % (epoch, tot[0], tot[3], tr_b, va_b, tr_b - va_b, marker))
    else:
        print("epoch %3d  g %.4f  d %.4f" % (epoch, tot[0], tot[3]))

    history.append(row)
    with open(log_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(history[0].keys()))
        w.writeheader(); w.writerows(history)

print("\nbest val buffered IoU %.4f at epoch %d, after %.1f minutes"
      % (best_iou, best_epoch, (time.time() - t0) / 60.0))

## 6. Curves

In [ ]:
h = {k: np.array([r[k] for r in history], float) for k in history[0]}
ev_ = ~np.isnan(h["val_buf"])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(h["epoch"], h["g_l1"], label="weighted L1")
if CFG["ADV_WEIGHT"] > 0:
    ax[0].plot(h["epoch"], h["g_adv"], label="adversarial")
    ax[0].plot(h["epoch"], h["d_loss"], label="discriminator")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("losses")

ax[1].plot(h["epoch"][ev_], h["train_buf"][ev_], label="train buffered")
ax[1].plot(h["epoch"][ev_], h["val_buf"][ev_], label="val buffered")
ax[1].plot(h["epoch"][ev_], h["val_strict"][ev_], "--", label="val strict")
ax[1].axvline(best_epoch, color="grey", lw=0.8, ls=":")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("IoU"); ax[1].legend(); ax[1].set_title(CFG["RUN_NAME"])
plt.tight_layout(); plt.savefig(OUT / "curves.png", dpi=130); plt.show()

## 7. Quick test check

From the best checkpoint, at the fixed training threshold and without TTA. For the reported
numbers run `02_evaluate.ipynb`.

In [ ]:
generator.load_weights(str(best_path))
train_strict, train_buf = quick_eval(train_eval_ds)
val_strict,   val_buf   = quick_eval(val_ds)
test_strict,  test_buf  = quick_eval(test_ds)

result = dict(CFG, test_strict=test_strict, test_buf=test_buf,
              val_strict=val_strict, val_buf=val_buf,
              train_strict=train_strict, train_buf=train_buf,
              gap_buf=train_buf - test_buf, best_epoch=best_epoch,
              minutes=history[-1]["minutes"], gen_params=int(generator.count_params()))
with open(OUT / "result.json", "w") as fh:
    json.dump(result, fh, indent=2)

print("%-14s %10s %10s" % ("split", "strict", "buffered"))
for name, s, b in (("train(%d)" % CFG["TRAIN_EVAL_N"], train_strict, train_buf),
                   ("val", val_strict, val_buf), ("test", test_strict, test_buf)):
    print("%-14s %10.4f %10.4f" % (name, s, b))
print("best epoch %d of %d" % (best_epoch, CFG["EPOCHS"]))

## 8. Samples

In [ ]:
n_show = 6
fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
for i, (src, tar) in enumerate(test_ds.take(n_show)):
    gen = generator(src, training=False)
    for j, (im, title) in enumerate(zip([src[0], gen[0], tar[0]],
                                        ["satellite", "generated", "target"])):
        axes[i, j].imshow(np.clip(im.numpy() * 0.5 + 0.5, 0, 1))
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(title)
plt.tight_layout(); plt.savefig(OUT / "samples.png", dpi=130); plt.show()
print("outputs in", OUT)